# 1. XGBoost on a synthetic invoice dataset

## Goal

Train XGBoost on the first 1,024 synthetic invoices and predict late payment and fraud for the same 128 held-out invoices used by SAP RPT 1.6. The main dataset has 187 fraud-labelled context rows and 22 fraud-labelled test rows. This notebook saves `data/synthetic_invoices/xgboost_results.csv` for the comparison notebook.

All outcomes are synthetic; the fixed split demonstrates model evaluation, not production performance.

## Setup: check the selected Python kernel

Run the next cell first. It installs the project requirements into this notebook's Python kernel only when pandas, scikit-learn, or XGBoost is missing.

In [1]:
import importlib.util
from pathlib import Path
import subprocess
import sys

missing = [name for name in ("pandas", "sklearn", "xgboost") if importlib.util.find_spec(name) is None]
if missing:
    project_root = next(path for path in (Path.cwd(), Path.cwd().parent) if (path / "requirements-experiment.txt").exists())
    print(f"Installing project requirements into {sys.executable}; missing: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(project_root / "requirements-experiment.txt")])
    importlib.invalidate_caches()
else:
    print("Required XGBoost packages are available in the selected kernel.")

Required XGBoost packages are available in the selected kernel.


## 1. Load the shared split

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

ROOT = next(path for path in (Path.cwd(), Path.cwd().parent) if (path / "data/synthetic_invoices/payment_behavior.csv").exists())
sys.path.insert(0, str(ROOT / "scripts"))
from generate_synthetic_invoices import CONTEXT_ROWS, PREDICTION_ROWS, TARGETS
from run_xgboost_invoices import DATA, load_split, train_models, predict_frame

train, test = load_split()
assert len(train) == CONTEXT_ROWS and len(test) == PREDICTION_ROWS
assert train.invoice_id.is_unique and test.invoice_id.is_unique
assert int(train.is_fraud.sum()) == 187 and int(test.is_fraud.sum()) == 22
print(f"Train/context: {len(train)} invoices ({int(train.is_fraud.sum())} fraud) | test: {len(test)} invoices ({int(test.is_fraud.sum())} fraud)")

Train/context: 1024 invoices (187 fraud) | test: 128 invoices (22 fraud)


## 2. Train, predict, and save

The runner excludes `invoice_id` and both targets from its feature matrix. It trains one classifier per target.

In [3]:
models, feature_columns = train_models(train)
results = predict_frame(models, feature_columns, test)
assert results.invoice_id.tolist() == test.invoice_id.tolist()
output_path = DATA / "xgboost_results.csv"
results.to_csv(output_path, index=False)
print(f"Saved {len(results)} predictions to {output_path.relative_to(ROOT)}")

Saved 128 predictions to data/synthetic_invoices/xgboost_results.csv


## 3. Review held-out results

In [4]:
metric_rows = []
for target in TARGETS:
    actual = results[f"actual_{target}"]
    predicted = results[f"predicted_{target}"]
    metric_rows.append({
        "Target": target,
        "Positive cases": int(actual.sum()),
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(actual, predicted, zero_division=0),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "ROC AUC": roc_auc_score(actual, results[f"probability_{target}"]),
    })
display(Markdown("### XGBoost test-set metrics (128 invoices)"))
display(pd.DataFrame(metric_rows).set_index("Target").round(3))
identification_rows = []
for target in TARGETS:
    actual = results[f"actual_{target}"]
    predicted = results[f"predicted_{target}"]
    identification_rows.append({
        "Target": target,
        "Actual cases": int(actual.sum()),
        "Correctly identified": int(((actual == 1) & (predicted == 1)).sum()),
        "False alarms": int(((actual == 0) & (predicted == 1)).sum()),
        "Total flagged": int(predicted.sum()),
    })
display(Markdown("### Cases identified and false alarms (128 test invoices)"))
display(pd.DataFrame(identification_rows).set_index("Target"))
display(Markdown("### Invoice-level predictions (first 8 test invoices)"))
display(results[["invoice_id", "actual_paid_late", "predicted_paid_late", "actual_is_fraud", "predicted_is_fraud"]].head(8))

### XGBoost test-set metrics (128 invoices)

,Positive cases,Accuracy,Precision,Recall,ROC AUC
Target,,,,,
paid_late,51,0.656,0.561,0.627,0.724
is_fraud,22,0.875,0.625,0.682,0.804


### Cases identified and false alarms (128 test invoices)

,Actual cases,Correctly identified,False alarms,Total flagged
Target,,,,
paid_late,51,32,25,57
is_fraud,22,15,9,24


### Invoice-level predictions (first 8 test invoices)

,invoice_id,actual_paid_late,predicted_paid_late,actual_is_fraud,predicted_is_fraud
1024,SYN-01025,1,0,0,0
1025,SYN-01026,0,0,1,1
1026,SYN-01027,0,0,0,0
1027,SYN-01028,0,0,0,0
1028,SYN-01029,1,0,0,0
1029,SYN-01030,0,0,0,0
1030,SYN-01031,0,0,0,0
1031,SYN-01032,0,0,0,0


## Next step

Run `rpt_btp_run.ipynb` after configuring your BTP deployment. Then open `compare_saved_results.ipynb`; it reads saved files and does not make an inference call.